# Atlas Notochord Cell Abundance Comparison

Compare raw embryo-level cell abundance trends across atlases for notochord-related cell types.

In [ ]:
suppressPackageStartupMessages({
  library(monocle3)
  library(BPCells)
  library(hooke)
  library(dplyr)
  library(tidyr)
  library(tibble)
  library(ggplot2)
  library(stringr)
  library(purrr)
})

hdf5_lib <- "/net/gs/vol3/software/modules-sw/hdf5/1.14.3/Linux/Ubuntu22.04/x86_64/lib/libhdf5.so.310"
if (file.exists(hdf5_lib)) dyn.load(hdf5_lib)

## Parameters

In [ ]:
temp_path <- "/net/trapnell/vol1/home/nlammers/tmp_files/nobackup/"
seahub_root <- "/net/seahub_zfish/vol1/data/reference_cds/"
datasets_to_compare <- c("v2.3.0", "v3.1.0")

out_dir <- "/Users/nick/Library/CloudStorage/GoogleDrive-nlammers@uw.edu/My Drive/projects/morphseq/results/20260615"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

hpf_range <- c(12, 72)
ctrl_labels <- c("EtOH", "DMSO", "ctrl-inj", "reference", "ctrl-uninj", "novehicle")

sample_group_candidates <- c("embryo_ID", "embryo_id", "embryo", "sample", "sample_id")
timepoint_candidates <- c("timepoint", "hpf", "stage_hpf", "age_hpf", "hours_post_fertilization")
cell_group_candidates <- c("cell_type", "cell_type_broad")

notochord_pattern <- regex("notochord|hypochord", ignore_case = TRUE)
theme_set(theme_bw(base_size = 12))

## Helpers

In [ ]:
first_present <- function(candidates, columns, label) {
  hit <- candidates[candidates %in% columns]
  if (length(hit) == 0) {
    stop(sprintf("Could not find %s. Tried: %s", label, paste(candidates, collapse = ", ")))
  }
  hit[[1]]
}

parse_hpf <- function(x) {
  if (is.numeric(x)) return(as.numeric(x))
  as.numeric(stringr::str_extract(as.character(x), "[0-9]+\\.?[0-9]*"))
}

resolve_cds_path <- function(dataset_name, root = seahub_root) {
  base_dir <- file.path(root, dataset_name)
  candidates <- unique(c(
    base_dir,
    file.path(base_dir, "projected_cds"),
    file.path(base_dir, paste0(dataset_name, "_projected_cds")),
    file.path(base_dir, paste0(dataset_name, "_projected_cds_", dataset_name)),
    file.path(base_dir, paste0("reference_projected_cds_", dataset_name))
  ))
  existing <- candidates[dir.exists(candidates)]
  if (length(existing) > 0) return(existing[[1]])

  if (dir.exists(base_dir)) {
    nested <- list.dirs(base_dir, recursive = TRUE, full.names = TRUE)
    nested <- nested[str_detect(basename(nested), regex("cds|monocle|projected", ignore_case = TRUE))]
    if (length(nested) > 0) return(nested[[1]])
  }

  stop(sprintf("Could not resolve CDS path for %s under %s", dataset_name, root))
}

load_atlas_notochord_counts <- function(dataset_name) {
  message("Loading ", dataset_name)
  cds_path <- resolve_cds_path(dataset_name)
  message("  CDS path: ", cds_path)

  cds <- load_monocle_objects(
    cds_path,
    matrix_control = list(matrix_class = "BPCells", matrix_path = temp_path)
  )

  md <- as.data.frame(colData(cds))
  sample_col <- first_present(sample_group_candidates, colnames(md), "embryo/sample column")
  time_col <- first_present(timepoint_candidates, colnames(md), "timepoint/hpf column")
  cell_col <- first_present(cell_group_candidates, colnames(md), "cell type column")

  md$hpf_numeric <- parse_hpf(md[[time_col]])
  keep_cells <- rownames(md)[!is.na(md$hpf_numeric) & md$hpf_numeric >= hpf_range[[1]] & md$hpf_numeric <= hpf_range[[2]]]

  if ("perturbation" %in% colnames(md)) {
    keep_cells <- intersect(
      keep_cells,
      rownames(md)[is.na(md$perturbation) | md$perturbation %in% ctrl_labels]
    )
  }

  if (length(keep_cells) == 0) {
    warning("No cells retained for ", dataset_name)
    return(tibble())
  }

  embryo_hpf <- md[keep_cells, , drop = FALSE] %>%
    mutate(embryo = as.character(.data[[sample_col]]), hpf = hpf_numeric) %>%
    filter(!is.na(embryo), !is.na(hpf)) %>%
    group_by(embryo) %>%
    summarise(hpf = median(unique(hpf), na.rm = TRUE), .groups = "drop")

  cds_keep <- cds[, keep_cells]

  ccs <- new_cell_count_set(
    cds_keep,
    sample_group = sample_col,
    cell_group = cell_col
  )

  count_mat <- counts(ccs) %>% as.matrix()
  notochord_cells <- rownames(count_mat)[str_detect(rownames(count_mat), notochord_pattern)]
  if (length(notochord_cells) == 0) {
    warning("No notochord-related cell groups found for ", dataset_name)
    return(tibble())
  }

  embryo_md <- embryo_hpf

  count_mat[notochord_cells, , drop = FALSE] %>%
    as.data.frame() %>%
    rownames_to_column("cell_group") %>%
    pivot_longer(-cell_group, names_to = "embryo", values_to = "cell_count") %>%
    left_join(embryo_md, by = "embryo") %>%
    mutate(
      dataset = dataset_name,
      hpf_label = factor(hpf, levels = sort(unique(hpf)))
    ) %>%
    filter(!is.na(hpf))
}

save_plot <- function(plot, filename, width = 11, height = 7) {
  pdf_path <- file.path(out_dir, paste0(filename, ".pdf"))
  png_path <- file.path(out_dir, paste0(filename, ".png"))
  ggsave(pdf_path, plot, width = width, height = height, units = "in")
  ggsave(png_path, plot, width = width, height = height, units = "in", dpi = 300)
  message("Saved: ", pdf_path)
  message("Saved: ", png_path)
}

## Load Atlas Counts

In [ ]:
notochord_counts <- purrr::map_dfr(datasets_to_compare, load_atlas_notochord_counts)

if (nrow(notochord_counts) == 0) {
  stop("No notochord count data found. Check dataset paths and cell type labels.")
}

notochord_counts <- notochord_counts %>%
  mutate(
    dataset = factor(dataset, levels = datasets_to_compare),
    cell_group = factor(cell_group, levels = sort(unique(cell_group))),
    hpf_label = factor(hpf, levels = sort(unique(hpf)))
  )

write.csv(
  notochord_counts,
  file.path(out_dir, "notochord_cell_counts_long.csv"),
  row.names = FALSE
)

notochord_counts %>%
  count(dataset, cell_group, hpf, name = "n_embryos") %>%
  arrange(cell_group, dataset, hpf)

## Per-Cell-Type Counts

In [ ]:
p_cell_counts <- ggplot(notochord_counts, aes(x = hpf_label, y = cell_count, fill = dataset)) +
  geom_boxplot(outlier.shape = NA, position = position_dodge(width = 0.75), width = 0.65) +
  geom_point(
    aes(color = dataset),
    position = position_jitterdodge(jitter.width = 0.16, dodge.width = 0.75),
    alpha = 0.35,
    size = 0.8,
    show.legend = FALSE
  ) +
  facet_wrap(~ cell_group, scales = "free_y") +
  scale_fill_brewer(palette = "Set2", drop = FALSE) +
  scale_color_brewer(palette = "Set2", drop = FALSE) +
  labs(
    title = "Notochord-related cell counts per embryo",
    subtitle = sprintf("%s-%s hpf; boxes shown only where a cell label exists in an atlas", hpf_range[[1]], hpf_range[[2]]),
    x = "hpf",
    y = "Cells per embryo",
    fill = "Atlas"
  ) +
  theme(
    legend.position = "bottom",
    axis.text.x = element_text(angle = 45, hjust = 1),
    panel.grid.minor = element_blank(),
    strip.text = element_text(size = 9)
  )

print(p_cell_counts)
save_plot(p_cell_counts, "notochord_cell_counts_by_atlas", width = 13, height = 8)

## Total Notochord-Related Counts

In [ ]:
notochord_totals <- notochord_counts %>%
  group_by(dataset, embryo, hpf, hpf_label) %>%
  summarise(total_notochord_cells = sum(cell_count, na.rm = TRUE), .groups = "drop")

write.csv(
  notochord_totals,
  file.path(out_dir, "notochord_total_counts_by_embryo.csv"),
  row.names = FALSE
)

p_total_counts <- ggplot(notochord_totals, aes(x = hpf_label, y = total_notochord_cells, fill = dataset)) +
  geom_boxplot(outlier.shape = NA, position = position_dodge(width = 0.75), width = 0.65) +
  geom_point(
    aes(color = dataset),
    position = position_jitterdodge(jitter.width = 0.16, dodge.width = 0.75),
    alpha = 0.4,
    size = 0.9,
    show.legend = FALSE
  ) +
  scale_fill_brewer(palette = "Set2", drop = FALSE) +
  scale_color_brewer(palette = "Set2", drop = FALSE) +
  labs(
    title = "Total notochord-related cells per embryo",
    subtitle = sprintf("%s-%s hpf", hpf_range[[1]], hpf_range[[2]]),
    x = "hpf",
    y = "Notochord-related cells per embryo",
    fill = "Atlas"
  ) +
  theme(
    legend.position = "bottom",
    axis.text.x = element_text(angle = 45, hjust = 1),
    panel.grid.minor = element_blank()
  )

print(p_total_counts)
save_plot(p_total_counts, "notochord_total_counts_by_atlas", width = 10, height = 6)

## Summary

In [ ]:
summary_table <- notochord_counts %>%
  group_by(dataset, cell_group, hpf) %>%
  summarise(
    n_embryos = n(),
    mean_count = mean(cell_count, na.rm = TRUE),
    median_count = median(cell_count, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(cell_group, dataset, hpf)

write.csv(summary_table, file.path(out_dir, "notochord_count_summary.csv"), row.names = FALSE)
summary_table